# Stage 4 — Unsloth SFT of Qwen3.8-27B (attacker + defender)

Fine-tunes the two arena policies on the Stage 3B datasets
(`attacker_train.json`, `defender_train.json`) and exports adapters / merged
16-bit / GGUF Q4_K_M (arena) / optional FP8 (vLLM).

**Hardware matrix** (Runtime → Change runtime type):

| GPU | VRAM | FP8 kernels | Stage 4 path |
|---|---|---|---|
| Tesla T4 | 16 GB | no | `precision: 4bit` (27B may OOM on one card; Kaggle 2xT4 works) |
| L4 | 24 GB | yes (sm89) | `precision: 4bit`; 27B FP8 does **not** fit |
| A100 | 40 GB | no (sm80) | `precision: 4bit`; `fp8` fails preflight |
| H100 | 80 GB | yes (sm90) | `precision: fp8` (TorchAO dynamic FP8) |

Never force fp16: Qwen3.8's gated delta net NaNs in pure float16 — Unsloth
picks the precision. `precision: auto` resolves at runtime.


In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    props = torch.cuda.get_device_properties(0)
    print(f"{torch.cuda.device_count()}x {props.name} sm{cap[0]}{cap[1]} "
          f"{props.total_memory / 1024**3:.1f} GB")
    print("FP8 capable:", cap >= (8, 9), "| 27B FP8 needs >= 48 GB")

## 1. Install the pinned Qwen3.8 stack

Per the Unsloth Qwen3.8 guide (2026-09): Transformers v5 (`qwen3_5`),
`trl==0.22.2`, `datasets==4.3.0`, TorchAO, Unsloth. Import order matters
(`unsloth` before `trl`); the package handles that.

In [ ]:
%%capture
!pip install -q --upgrade uv
!uv pip install -qqq unsloth triton "huggingface_hub>=0.34.0" "datasets==4.3.0"
!uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"
!uv pip install -qqq "transformers==5.15.1"
!uv pip install -qqq --no-deps "trl==0.22.2"
!pip install -q pyyaml

## 2. Point the notebook at the repo and the Stage 3B data

Either clone the repo (`SFT_REPO_URL`) or set `SFT_REPO` to a Drive copy.
Then mount Drive with `attacker_train.json` / `defender_train.json`
(or upload them) so the files land in the repo root.

In [ ]:
import os, subprocess, sys

REPO = os.environ.get("SFT_REPO", "/content/RL_cyberbench")
REPO_URL = os.environ.get("SFT_REPO_URL", "")
if not os.path.isdir(os.path.join(REPO, "SFT_pentest_network")) and REPO_URL:
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
assert os.path.isdir(os.path.join(REPO, "SFT_pentest_network")), (
    "Set SFT_REPO to the repo checkout (or SFT_REPO_URL to clone it)")
os.chdir(REPO)
%env PYTHONPATH={REPO}/SFT_pentest_network
print("repo:", REPO)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DATA = os.environ.get("SFT_DATA", "/content/drive/MyDrive/pentest_sft")

import glob, shutil
for name in ("attacker_train.json", "defender_train.json", "sft_report.json"):
    src = os.path.join(DATA, name)
    if os.path.isfile(src):
        shutil.copy(src, os.path.join(REPO, name))
        print("copied", name)
    else:
        print("not found (upload it):", src)
for name in ("attacker_train.json", "defender_train.json"):
    path = os.path.join(REPO, name)
    print(name, "->", os.path.getsize(path) if os.path.isfile(path) else "MISSING")

## 3. Preflight + data validation

In [ ]:
!python -m sft.train --config SFT_pentest_network/configs/sft.default.yaml --dry-run
!python -m sft.data --config SFT_pentest_network/configs/sft.default.yaml --validate --stats

## 4. Train

`--smoke` runs 8 records / 2 steps to validate the full path first.
The default config trains **both** agents sequentially on one GPU and exports
GGUF Q4_K_M. For merged 16-bit / FP8 exports, set
`export.merged_16bit` / `export.fp8` in the YAML *before* the full run
(FP8 requires `merged_16bit`).

In [ ]:
!python -m sft.train --config SFT_pentest_network/configs/sft.default.yaml --agent attacker --smoke

In [ ]:
!python -m sft.train --config SFT_pentest_network/configs/sft.default.yaml --agent both
# FP8 on an H100 (80 GB, sm90):
# !python -m sft.train --config SFT_pentest_network/configs/sft.default.yaml --agent both --precision fp8

## 5. Verify (trained != initial, attacker != defender)

In [ ]:
!python -m sft.verify --all --output-root sft_runs
!python -m sft.verify --compare sft_runs/attacker/adapters sft_runs/defender/adapters

## 6. Collect artifacts

Adapters + manifests are small enough for Drive. GGUFs are ~15-16 GB each:
copy them to Drive (or push to HF Hub in `run.json` via `export.push_to_hub`).

In [ ]:
import glob, os, shutil, tarfile
os.makedirs(DATA, exist_ok=True)
for pattern in ("sft_runs/*/adapters", "sft_runs/*/run.json", "sft_runs/*/verify.json"):
    for path in glob.glob(pattern):
        dest = os.path.join(DATA, path)
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        if os.path.isdir(path):
            shutil.copytree(path, dest, dirs_exist_ok=True)
        else:
            shutil.copy(path, dest)
for gguf in glob.glob("sft_runs/*/gguf/**/*.gguf", recursive=True):
    print("GGUF:", gguf, f"{os.path.getsize(gguf) / 1024**3:.1f} GiB -> copy to Drive manually")

archive = "/content/sft_adapters.tgz"
with tarfile.open(archive, "w:gz") as tar:
    tar.add("sft_runs")
from google.colab import files
files.download(archive)

## 7. Arena promotion (run locally after copying the GGUF)

```bash
make sft-verify SFT_RUNS=sft_runs
PYTHONPATH=SFT_pentest_network .venv/bin/python -m sft.promote \
  --gguf models/sft/attacker-XXXXXXXX.gguf --agent attacker \
  --seed-volume --restart-sidecar
```

Then re-run `make lab-up` (or the next collection round) so `llama-server`
serves the fine-tuned weights. Stage 5 GRPO warm-starts from
`sft_runs/<agent>/adapters`.